# 📊 Data, Calculations, Actions

## Grokking Simplicity für LLM-Programmierung

Das Buch *Grokking Simplicity* teilt Code in drei Kategorien:

| Kategorie | Was ist das? | Beispiel in DSPy |
|-----------|-------------|------------------|
| **DATA** | Reine Deklarationen, kein Verhalten | Signatures, Datasets |
| **CALCULATIONS** | Reine Funktionen, gleicher Input = gleicher Output | Metriken |
| **ACTIONS** | Seiteneffekte, I/O, Netzwerk | LLM-Aufrufe |

Warum ist das wichtig? Weil DATA und CALCULATIONS **sofort testbar** sind — ohne API-Key, ohne Wartezeit, ohne Kosten. Nur ACTIONS brauchen das echte Modell.

In [3]:
import sys
sys.path.insert(0, ".")
import dspy
from dspy_tasks.tasks import get_task, list_by_tier
from dspy_tasks.data import ClassifySentiment, ExtractEntities, SummarizeText
from dspy_tasks.calculations import sentiment_exact_match, entity_f1, summary_quality
from dspy_tasks.actions import run_baseline
from dspy_tasks.visualize import *

In [ ]:
from dspy_tasks.visualize import mermaid

mermaid("""
graph LR
    subgraph "DATA 📦"
        S["Signatures<br/>(was rein/raus geht)"]
        D["Datasets<br/>(Beispiele)"]
    end
    subgraph "CALCULATIONS 🧮"
        M["Metriken<br/>(Qualität messen)"]
    end
    subgraph "ACTIONS ⚡"
        L["LLM-Aufrufe<br/>(Seiteneffekte!)"]
        E["Evaluation<br/>(Ergebnisse)"]
    end
    S --> L
    D --> M
    L --> M
    M --> E
    style S fill:#e8f0fe,stroke:#0078d4
    style D fill:#e8f0fe,stroke:#0078d4
    style M fill:#f0fdf0,stroke:#107c10
    style L fill:#fdf6f0,stroke:#ca5010
    style E fill:#fdf6f0,stroke:#ca5010
""")

## Teil 1: DATA — Signatures als Deklarationen

Eine Signature ist **reines Data** — sie beschreibt was du willst, hat aber kein Verhalten. Du könntest sie als JSON serialisieren und übers Netzwerk schicken. Keine Magie, keine LLM-Aufrufe.

In [4]:
# A Signature is pure DATA — it declares intent, not implementation
print("ClassifySentiment fields:")
for name, field in ClassifySentiment.model_fields.items():
    field_info = ClassifySentiment.__annotations__.get(name)
    print(f"  {name}: {field_info}")

display_insight("DATA-Prinzip",
    "Signatures sind reine Daten-Deklarationen. Sie beschreiben WAS du willst, nie WIE. "
    "Du könntest sie als JSON serialisieren — sie haben kein Verhalten.")

ClassifySentiment fields:
  review: <class 'str'>
  sentiment: <class 'str'>


In [5]:
task = get_task("sentiment")
examples = task.load_examples()
print(f"Dataset: {len(examples)} examples\n")
for ex in examples[:5]:
    print(f"  Review: {str(ex.review)[:60]}...")
    print(f"  Sentiment: {ex.sentiment}\n")

Dataset: 25 examples

  Review: Absolutely love this blender! It crushes ice in seconds and ...
  Sentiment: positive

  Review: The laptop arrived with a cracked screen and customer suppor...
  Sentiment: negative

  Review: The coffee maker works as described. Nothing exceptional, no...
  Sentiment: neutral

  Review: Oh great, another pair of wireless earbuds that dies after t...
  Sentiment: negative

  Review: I was skeptical at first, but this standing desk has complet...
  Sentiment: positive



## Teil 2: CALCULATIONS — Metriken als reine Funktionen

Metriken sind **Calculations** — gleicher Input, immer gleicher Output. Du kannst sie mit Fake-Daten testen, ganz ohne API-Key. Sie sind unendlich schnell und perfekt deterministisch.

Das hier ist deine **Spezifikation** — deine "Test-Suite" für KI.

In [6]:
# CALCULATION — No LLM needed! Instantly testable.
class FakeExample:
    sentiment = "positive"
class FakePrediction:
    sentiment = "positive"
class WrongPrediction:
    sentiment = "negative"

score_correct = sentiment_exact_match(FakeExample(), FakePrediction())
score_wrong = sentiment_exact_match(FakeExample(), WrongPrediction())

print(f"Correct prediction score: {score_correct}")   # 1.0
print(f"Wrong prediction score:   {score_wrong}")     # 0.0

display_insight("CALCULATION-Prinzip",
    "Metriken sind reine Funktionen. Du kannst sie mit Fake-Daten testen, "
    "kein API-Key nötig. Sie sind unendlich schnell und perfekt deterministisch. "
    "Das hier ist deine Spezifikation — deine 'Test-Suite' für KI.")

Correct prediction score: 1.0
Wrong prediction score:   0.0


In [7]:
class ExampleEntities:
    entities = "Apple, Google, Microsoft"
class PredEntities:
    entities = "Apple, Microsoft, Amazon"  # Got 2 of 3, added 1 wrong

score = entity_f1(ExampleEntities(), PredEntities())
print(f"Entity F1 score: {score:.3f}")
print("(2 correct out of 3 expected, 1 false positive → F1 reflects both precision and recall)")

Entity F1 score: 0.667
(2 correct out of 3 expected, 1 false positive → F1 reflects both precision and recall)


## Teil 3: ACTIONS — LLM-Aufrufe als I/O

Jetzt wird's ernst: **Actions** sind der Teil mit Seiteneffekten. Sie rufen echte Modelle auf, brauchen Zeit, kosten Geld, und können fehlschlagen. Wir trennen sie, damit alles andere testbar bleibt.

In [8]:
import ipywidgets as widgets
from dspy_tasks.config import get_available_models, get_default_model, configure_dspy

AVAILABLE_MODELS = get_available_models()

model_dd = model_picker(AVAILABLE_MODELS, default=get_default_model())
btn = run_button("Run Sentiment Baseline")
out = widgets.Output()

def on_run(b):
    with out:
        out.clear_output()
        print(f"⏳ Running sentiment task on {model_dd.value}...")
        result = run_baseline("sentiment", model_dd.value, max_eval=10)
        display_score("Baseline Score", result.score)
        print(f"⏱️  {result.elapsed_seconds}s | {result.llm_calls} LLM calls")
        display_results_table(result.individual_scores)

btn.on_click(on_run)
display(widgets.HBox([model_dd, btn]), out)

Output()

In [9]:
# Run all 3 Tier 1 tasks
task_dd = widgets.Dropdown(
    options=[(t.name, t.id) for t in list_by_tier(1)[:3]],
    description="Task:"
)
btn2 = run_button("Run Task")
out2 = widgets.Output()

def on_run2(b):
    with out2:
        out2.clear_output()
        task = get_task(task_dd.value)
        print(f"⏳ Running {task.name} on {model_dd.value}...")
        result = run_baseline(task_dd.value, model_dd.value, max_eval=8)
        display_score(task.name, result.score)
        print(f"⏱️  {result.elapsed_seconds}s | 💡 {task.teaching_point}")
        display_results_table(result.individual_scores[:5])

btn2.on_click(on_run2)
display(widgets.HBox([task_dd, btn2]), out2)

Output()

## ⏭️ Weiter geht's!

Du hast DATA, CALCULATIONS und ACTIONS sauber getrennt. Aber was passiert, wenn wir das **Modul tiefer** machen?

`dspy.Predict` macht einen einzigen LLM-Aufruf. `dspy.ChainOfThought` fügt automatisch einen Denkschritt hinzu. `dspy.ReAct` kann sogar Tools nutzen. Und alle drei haben **das gleiche Interface**!

Das ist das Thema von *A Philosophy of Software Design* — und Notebook 02.

In [10]:
display_insight("Das Fundament",
    "Signatures und Metriken sind dein Quellcode — rein, testbar und portabel. "
    "LLM-Aufrufe sind nur die I/O-Schicht. Wenn du das Modell wechselst, "
    "ändern sich nur die ACTIONS. DATA und CALCULATIONS bleiben gleich.",
    icon="🏗️")